In [1]:
import wandb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def sum_from_dict_list(config, keys):
    total = 0
    for key in keys:
        value = config.get(key, [])
        if isinstance(value, dict) and 'value' in value:
            value = value['value']
        if not isinstance(value, list):
            if isinstance(value, dict) and 'params' in value:
                value = value['params'].get(key, [])
        if not isinstance(value, list):
            continue
        total += sum([int(v) for v in value])
    return total

def get_embedding_product(config):
    containers = [config]
    if isinstance(config.get('model'), dict):
        containers.append(config['model'])
        if isinstance(config['model'].get('value'), dict):
            containers.append(config['model']['value'])
        if isinstance(config['model'].get('params'), dict):
            containers.append(config['model']['params'])
    if isinstance(config.get('tokenizer'), dict):
        containers.append(config['tokenizer'])
        if isinstance(config['tokenizer'].get('value'), dict):
            containers.append(config['tokenizer']['value'])
        if isinstance(config['tokenizer'].get('params'), dict):
            containers.append(config['tokenizer']['params'])

    embedding_dim = None
    max_vocab_size = None
    for container in containers:
        if embedding_dim is None and 'embedding_dim' in container:
            try:
                embedding_dim = int(container['embedding_dim'])
            except Exception:
                pass
        if max_vocab_size is None and 'max_vocab_size' in container:
            try:
                max_vocab_size = int(container['max_vocab_size'])
            except Exception:
                pass
        if 'params' in container and isinstance(container['params'], dict):
            if embedding_dim is None and 'embedding_dim' in container['params']:
                try:
                    embedding_dim = int(container['params']['embedding_dim'])
                except Exception:
                    pass
            if max_vocab_size is None and 'max_vocab_size' in container['params']:
                try:
                    max_vocab_size = int(container['params']['max_vocab_size'])
                except Exception:
                    pass
    if embedding_dim and max_vocab_size:
        return embedding_dim * max_vocab_size
    return 0

def fetch_project_data(project_path, model_names):
    try:
        api = wandb.Api()
        runs = api.runs(project_path)

        data = []
        for run in runs:
            if run.state not in ["finished", "running"]:
                continue

            config = run.config
            summary = run.summary._json_dict

            model_name = None
            if 'model' in config:
                model_config = config['model']
                if isinstance(model_config, dict):
                    if 'value' in model_config:
                        value = model_config['value']
                        if isinstance(value, dict):
                            model_name = value.get('name')
                        else:
                            model_name = value
                    elif 'name' in model_config:
                        model_name = model_config.get('name')
                else:
                    model_name = model_config
            if model_name not in model_names:
                continue

            train_accuracy = None
            for path in ['train/metric/accuracy', 'train_accuracy', 'accuracy', 'train/accuracy']:
                if path in summary:
                    train_accuracy = summary[path]
                    break

            node_count = None
            if model_name == 'UnsyncedRecurrentDifflogic':
                dicts = []
                if isinstance(config.get('model'), dict):
                    dicts.append(config['model'])
                    if isinstance(config['model'].get('value'), dict):
                        dicts.append(config['model']['value'])
                    if isinstance(config['model'].get('params'), dict):
                        dicts.append(config['model']['params'])
                dicts.append(config)
                for d in dicts:
                    node_count = sum_from_dict_list(d, [
                        'k_layers_sizes', 'l_layers_sizes', 'm_layers_sizes',
                        'n_layers_sizes', 'p_layers_sizes'
                    ])
                    if node_count > 0:
                        break
            elif model_name in ['UnsyncedGRU', 'UnsyncedRNN']:
                total_params = None
                if 'model' in config:
                    model_config = config['model']
                    if isinstance(model_config, dict):
                        if 'value' in model_config and isinstance(model_config['value'], dict):
                            total_params = model_config['value'].get('total_trainable_params')
                        else:
                            total_params = model_config.get('total_trainable_params')
                    if total_params is None and 'total_trainable_params' in model_config:
                        total_params = model_config['total_trainable_params']
                if total_params is None and 'total_trainable_params' in config:
                    total_params = config['total_trainable_params']
                try:
                    total_params = int(total_params)
                except Exception:
                    total_params = None

                emb_prod = get_embedding_product(config)
                node_count = None
                if total_params is not None and emb_prod is not None:
                    node_count = total_params - emb_prod

            if node_count is None or node_count <= 0 or not np.isfinite(node_count):
                continue
            if train_accuracy is None or not np.isfinite(train_accuracy):
                continue

            data.append({
                'nodes': int(node_count),
                'train_accuracy': float(train_accuracy),
                'run_id': run.id,
                'model': model_name
            })

        return pd.DataFrame(data)

    except Exception as e:
        print(f"Error fetching project {project_path}: {e}")
        return pd.DataFrame()

def filter_monotonic_increasing(df):
    filtered_data = []
    for model_name, group in df.groupby('model'):
        sorted_group = group.sort_values('nodes')
        max_accuracy = -1
        valid_points = []
        for _, row in sorted_group.iterrows():
            if row['train_accuracy'] >= max_accuracy:
                max_accuracy = row['train_accuracy']
                valid_points.append(row)
        filtered_data.extend(valid_points)
    return pd.DataFrame(filtered_data)

def perform_log_linear_regression(x, y):
    if len(x) < 2:
        return None, None, None, None
    x_np = np.array(x)
    y_np = np.array(y)
    mask = (x_np > 0) & np.isfinite(x_np) & np.isfinite(y_np)
    x_np = x_np[mask]
    y_np = y_np[mask]
    if len(x_np) < 2:
        return None, None, None, None
    log_x = np.log10(x_np)
    X = log_x.reshape(-1, 1)
    model = LinearRegression()
    model.fit(X, y_np)
    y_pred = model.predict(X)
    r2 = r2_score(y_np, y_pred)
    return model.coef_[0], model.intercept_, r2, model

def setup_publication_style():
    style = {
        'UnsyncedGRU': {
            'color': '#1f77b4',    # blue
            'marker': 'o',
            'hatch': '//',
            'linestyle': '-'
        },
        'UnsyncedRNN': {
            'color': '#2ca02c',    # green
            'marker': 's',
            'hatch': '\\\\',
            'linestyle': '--'
        },
        'UnsyncedRecurrentDifflogic': {
            'color': '#9467bd',    # purple
            'marker': 'D',
            'hatch': 'xx',
            'linestyle': ':'
        },
        'default': {
            'color': '#444444',
            'marker': '^',
            'hatch': '||',
            'linestyle': '-.'
        }
    }
    plt.rcParams.update({
        'font.size': 9,
        'axes.labelsize': 10,
        'legend.fontsize': 8,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'font.family': 'serif',
        'axes.grid': True,
        'axes.axisbelow': True,
        'grid.alpha': 0.3,
        'grid.linewidth': 0.8,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.spines.left': True,
        'axes.spines.bottom': True,
        'axes.linewidth': 1.0,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'lines.markersize': 6,
        'lines.linewidth': 2,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.titlepad': 4,
    })
    return style

# ... (rest of the code unchanged above)

def create_scatter_plot(data, output_dir=".", save_plots=True):
    style = setup_publication_style()
    fig, ax = plt.subplots(figsize=(3.25, 2.4), dpi=300)

    filtered_data = filter_monotonic_increasing(data)
    model_label_map = {
        'UnsyncedGRU': 'GRU',
        'UnsyncedRNN': 'RNN',
        'UnsyncedRecurrentDifflogic': 'RDDLGN'
    }

    regression_results = {}
    handles = []
    for name, group in filtered_data.groupby('model'):
        valid = (group['nodes'] > 0) & np.isfinite(group['nodes']) & np.isfinite(group['train_accuracy'])
        if not valid.any():
            print(f"Skipping {name} (no valid node counts).")
            continue
        group = group[valid]
        st = style.get(name, style['default'])
        nodes_millions = group['nodes'] / 1e6
        accuracies = group['train_accuracy'] * 100

        h = ax.scatter(
            nodes_millions, accuracies,
            color=st['color'],
            marker=st['marker'],
            label=model_label_map.get(name, name),
            edgecolors='black',
            linewidths=0.8,
            alpha=0.9,
            zorder=3
        )
        handles.append(h)

        slope, intercept, r2, model = perform_log_linear_regression(nodes_millions, accuracies)
        if slope is not None:
            x_range = np.logspace(np.log10(nodes_millions.min()), np.log10(nodes_millions.max()), 100)
            log_x = np.log10(x_range)
            y_pred = model.predict(log_x.reshape(-1, 1))
            ax.plot(
                x_range, y_pred,
                color=st['color'],
                linestyle=st['linestyle'],
                linewidth=1.3,
                alpha=0.8,
                zorder=2
            )
            ax.fill_between(
                x_range, y_pred - 0.1, y_pred + 0.1,
                facecolor='none', edgecolor=st['color'],
                hatch=st['hatch'],
                linewidth=0, zorder=1, alpha=0.15
            )
            regression_results[name] = {
                'slope': slope,
                'intercept': intercept,
                'r2': r2
            }

    ax.set_xlabel('Node Count (Millions)', fontweight='bold', labelpad=2, fontsize=9)
    ax.set_ylabel('Training Accuracy (%)', fontweight='bold', labelpad=2, fontsize=9)
    ax.set_xscale('log')
    ax.set_ylim(0, None)
    # Adjusted x-axis as requested:
    ax.set_xlim(left=1e4/1e6, right=5e7/1e6)

    leg = ax.legend(
        handles=handles,
        labels=[model_label_map.get(n, n) for n in filtered_data['model'].unique() if (filtered_data['model'] == n).any()],
        frameon=False,
        loc='lower right',
        ncol=1,
        borderaxespad=0.3,
        handletextpad=0.5,
        columnspacing=0.7
    )
    plt.tight_layout(pad=0.25)
    if save_plots:
        pdf_path = os.path.join(output_dir, 'nodes_train_accuracy_loglinear_aaai_halfpage.pdf')
        fig.savefig(pdf_path, format='pdf', dpi=300, bbox_inches='tight', pad_inches=0.03)
        png_path = os.path.join(output_dir, 'nodes_train_accuracy_loglinear_aaai_halfpage.png')
        fig.savefig(png_path, format='png', bbox_inches='tight', pad_inches=0.03)

        print(f"Plot saved as {pdf_path}")

    plt.close(fig)
    return fig, regression_results

# ... (rest of the code unchanged below)

def print_caption():
    print(
        "Figure: Training accuracy as a function of node count (in millions). "
        "Trend lines and data points are shown for GRU, RNN, and RDDLGN models. "
        "Distinct marker shapes, hatches, and colors ensure interpretability in both color and black-and-white."
    )

def print_summary_statistics(data, regression_results):
    print("\n" + "="*80)
    print("NODE COUNT VS TRAINING ACCURACY ANALYSIS (LOG-LINEAR)")
    print("="*80)
    for name, group in data.groupby('model'):
        print(f"\n{name}:")
        print(f"  Number of runs: {len(group)}")
        print(f"  Nodes: {group['nodes'].min()/1e6:.2f}M - {group['nodes'].max()/1e6:.2f}M")
        print(f"  Accuracy: {group['train_accuracy'].min()*100:.2f}% - {group['train_accuracy'].max()*100:.2f}%")
    print("\nLOG-LINEAR REGRESSION RESULTS:")
    for name, results in regression_results.items():
        print(f"\n{name}:")
        print(f"  Slope: {results['slope']:.6f} (accuracy change per log10M nodes)")
        print(f"  Intercept: {results['intercept']:.2f}%")
        print(f"  R²: {results['r2']:.4f}")
    print("="*80)

def save_data_to_csv(data, output_dir):
    if len(data) > 0:
        path = f"{output_dir}/nodes_train_accuracy_log_linear.csv"
        data.to_csv(path, index=False)
        print(f"Data saved to {path}")

def main():
    print("="*80)
    print("NODE COUNT VS TRAINING ACCURACY COMPARISON (LOG-LINEAR)")
    print("="*80)

    project_path = "sbuehrer-eth-z-rich/Architecture"
    model_names = ['UnsyncedGRU', 'UnsyncedRNN', 'UnsyncedRecurrentDifflogic']

    output_dir = "nodes_accuracy_analysis"
    os.makedirs(output_dir, exist_ok=True)

    print("Fetching project data...")
    data = fetch_project_data(project_path, model_names)

    if len(data) == 0:
        print("No valid data found. Please check project path and model names.")
        return

    print("\nCreating AAAI half-page width plot...")
    fig, regression_results = create_scatter_plot(data, output_dir)

    print_caption()
    print_summary_statistics(data, regression_results)
    save_data_to_csv(data, output_dir)
    print(f"\nAll outputs saved to: {output_dir}/")

if __name__ == "__main__":
    main()